# Etapa 4 — Aprendizado probabilístico e perfis ocultos

**Fundamentos de Inteligência Artificial | Focos de queimadas na Amazônia**

Implementação própria de **MLE, MAP, Bernoulli Naive Bayes e Expectation-Maximization (EM)**, com comparação controlada entre Gaussian Naive Bayes e misturas gaussianas por classe.

**Protocolo:** utilizar exclusivamente dados de desenvolvimento de **2024**; avaliação nos mesmos cinco folds espaciais; imputação, padronização, discretização e limiar de FRP elevada calculados **somente no treino** de cada fold. O conjunto de 2025 permanece reservado à avaliação final do projeto.

**Dependências locais:** `utils.py`, `aprendizado_probabilistico.py` e `em.py` precisam estar ao lado deste notebook. Não há necessidade de copiar as implementações desses módulos para dentro das células.

**Execução:** execute as células em ordem a partir de um kernel reiniciado. As figuras são salvas na pasta de trabalho do kernel, inclusive quando o renderizador do VS Code falha.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)
from sklearn.naive_bayes import BernoulliNB  # Apenas verificação da implementação própria

from utils import carregar_dados, iterar_folds, preparar_particao
from aprendizado_probabilistico import (
    mle_bernoulli, map_bernoulli, BernoulliNaiveBayes,
)
from em import NaiveBayesComEM

np.random.seed(42)
DIRETORIO_GRAFICOS = Path("graficos_etapa4")
DIRETORIO_GRAFICOS.mkdir(exist_ok=True)
dados = carregar_dados(2024)
print(f"Registros de desenvolvimento (2024): {len(dados):,}")
print("Atributos:", ["numero_dias_sem_chuva", "precipitacao", "lat", "lon", "mes", "hora_utc"])

Registros de desenvolvimento (2024): 140,346
Atributos: ['numero_dias_sem_chuva', 'precipitacao', 'lat', 'lon', 'mes', 'hora_utc']


## 1. Estimativas MLE e MAP

Para $y_i\in\{0,1\}$, $\hat\theta_{MLE}=\sum_i y_i/n$. Com priori $\mathrm{Beta}(\alpha,\beta)$, a moda posterior interior é $\hat\theta_{MAP}=(k+\alpha-1)/(n+\alpha+\beta-2)$, quando ambos os parâmetros posteriores são maiores que um.

In [2]:
# Teste verificável com 3 sucessos em 4 observações
y_exemplo = np.array([1, 1, 1, 0])
print('MLE:', mle_bernoulli(y_exemplo))
print('MAP Beta(2,2):', map_bernoulli(y_exemplo, 2, 2))
assert np.isclose(mle_bernoulli(y_exemplo), 0.75)
assert np.isclose(map_bernoulli(y_exemplo, 2, 2), 2/3)

MLE: 0.75
MAP Beta(2,2): 0.6666666666666666


### Aplicação ao domínio

Estimamos a frequência da classe `FRP elevada` apenas com os rótulos do treino. O corte de FRP elevada é o percentil 75 do próprio treino, calculado por `preparar_particao()`. Os folds mantêm o mesmo protocolo do restante da equipe.

In [3]:
dados = carregar_dados(2024)
resultados_estimadores = []
for fold, treino, validacao in iterar_folds(dados, 'espacial'):
    p = preparar_particao(treino, validacao, 'discretizada')
    y = p.y_cls_treino.to_numpy()
    resultados_estimadores.append({'fold': fold, 'n_treino': len(y), 'limiar_frp': p.limiar_frp,
                                   'positivos': int(y.sum()), 'MLE': mle_bernoulli(y),
                                   'MAP_Beta_2_2': map_bernoulli(y, 2, 2)})
tabela_estimadores = pd.DataFrame(resultados_estimadores)
display(tabela_estimadores)

,fold,n_treino,limiar_frp,positivos,MLE,MAP_Beta_2_2
0,0,112277,59.5,28105,0.250318,0.250323
1,1,112277,58.0,28092,0.250203,0.250207
2,2,112276,58.6,28079,0.250089,0.250094
3,3,112277,57.9,28084,0.250131,0.250136
4,4,112277,58.1,28104,0.250310,0.250314


In [4]:
# Efeito da priori em amostras pequenas (apenas ilustração, sem usar validação)
y_pequeno = np.array([1, 1, 0, 1, 0])
experimento_priori = pd.DataFrame([{'estimador': 'MLE', 'theta': mle_bernoulli(y_pequeno)},
                                   {'estimador': 'MAP Beta(2,2)', 'theta': map_bernoulli(y_pequeno, 2, 2)},
                                   {'estimador': 'MAP Beta(5,5)', 'theta': map_bernoulli(y_pequeno, 5, 5)}])
display(experimento_priori)

,estimador,theta
0,MLE,0.600000
1,"MAP Beta(2,2)",0.571429
2,"MAP Beta(5,5)",0.538462


## 2. Naive Bayes de Bernoulli do zero

A implementação em `aprendizado_probabilistico.py` estima as probabilidades condicionais dos atributos binários com suavização aditiva e calcula as probabilidades posteriores no espaço logarítmico. A classe 1 significa FRP superior ou igual ao Q75 do treino.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
linhas = []
for fold, treino, validacao in iterar_folds(dados, 'espacial'):
    p = preparar_particao(treino, validacao, 'discretizada')
    Xtr, Xva = p.X_treino.to_numpy(), p.X_avaliacao.to_numpy()
    ytr, yva = p.y_cls_treino.to_numpy(), p.y_cls_avaliacao.to_numpy()
    modelo = BernoulliNaiveBayes(alpha=1.0).fit(Xtr, ytr)
    pred = modelo.predict(Xva)
    linhas.append({'fold': fold, 'limiar_frp': p.limiar_frp,
                   'acuracia': accuracy_score(yva, pred),
                   'precisao': precision_score(yva, pred, zero_division=0),
                   'recall': recall_score(yva, pred, zero_division=0),
                   'f1': f1_score(yva, pred, zero_division=0),
                   'tn': confusion_matrix(yva, pred, labels=[0,1])[0,0],
                   'fp': confusion_matrix(yva, pred, labels=[0,1])[0,1],
                   'fn': confusion_matrix(yva, pred, labels=[0,1])[1,0],
                   'tp': confusion_matrix(yva, pred, labels=[0,1])[1,1]})
tabela_nb = pd.DataFrame(linhas)
display(tabela_nb)
display(tabela_nb[['acuracia','precisao','recall','f1']].agg(['mean','std']))

,fold,limiar_frp,acuracia,precisao,recall,f1,tn,fp,fn,tp
0,0,59.5,0.773594,0.0,0.0,0.0,21714,0,6355,0
1,1,58.0,0.740710,0.0,0.0,0.0,20791,0,7278,0
2,2,58.6,0.753260,0.0,0.0,0.0,21144,0,6926,0
3,3,57.9,0.738181,0.0,0.0,0.0,20720,0,7349,0
4,4,58.1,0.743382,0.0,0.0,0.0,20866,0,7203,0


,acuracia,precisao,recall,f1
mean,0.749825,0.0,0.0,0.0
std,0.014465,0.0,0.0,0.0


### 2.1 Diagnóstico do Bernoulli Naive Bayes no primeiro fold

A comparação com `BernoulliNB` do scikit-learn é **somente uma checagem numérica**: o modelo entregue continua sendo a implementação própria. Como o limiar 0,5 pode resultar em nenhuma previsão positiva, examinamos as probabilidades e a ROC-AUC.

In [6]:

from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import roc_auc_score

# Diagnóstico no primeiro fold espacial
fold, treino, validacao = next(iter(iterar_folds(dados, 'espacial')))
p = preparar_particao(treino, validacao, 'discretizada')

Xtr = p.X_treino.to_numpy()
Xva = p.X_avaliacao.to_numpy()
ytr = p.y_cls_treino.to_numpy()
yva = p.y_cls_avaliacao.to_numpy()

# Implementação própria
nb_proprio = BernoulliNaiveBayes(alpha=1.0).fit(Xtr, ytr)
prob_proprio = nb_proprio.predict_proba(Xva)[:, 1]

# Implementação de referência
nb_sklearn = BernoulliNB(alpha=1.0).fit(Xtr, ytr)
prob_sklearn = nb_sklearn.predict_proba(Xva)[:, 1]

print("=== VERIFICAÇÃO DA IMPLEMENTAÇÃO ===")
print("Diferença máxima:",
      np.max(np.abs(prob_proprio - prob_sklearn)))

print("\n=== PROBABILIDADES DA CLASSE 1 ===")
print(pd.Series(prob_proprio).describe(
    percentiles=[.01, .10, .25, .50, .75, .90, .99]
))

print("\n=== CLASSIFICAÇÕES ===")
print("Positivos reais:", int(yva.sum()))
print("Previstos com limiar 0.5:",
      int((prob_proprio >= 0.5).sum()))

print("\n=== CAPACIDADE DE DISCRIMINAÇÃO ===")
print("ROC-AUC:", roc_auc_score(yva, prob_proprio))

print("\n=== PROBABILIDADE POR CLASSE REAL ===")
print(pd.DataFrame({
    "classe_real": yva,
    "probabilidade": prob_proprio
}).groupby("classe_real")["probabilidade"].agg(
    ["mean", "median", "min", "max"]
))


=== VERIFICAÇÃO DA IMPLEMENTAÇÃO ===
Diferença máxima: 5.664103397140519e-06

=== PROBABILIDADES DA CLASSE 1 ===
count    28069.000000
mean         0.233380
std          0.079820
min          0.134006
1%           0.134006
10%          0.139166
25%          0.139166
50%          0.238572
75%          0.300690
90%          0.314740
99%          0.384680
max          0.395089
dtype: float64

=== CLASSIFICAÇÕES ===
Positivos reais: 6355
Previstos com limiar 0.5: 0

=== CAPACIDADE DE DISCRIMINAÇÃO ===
ROC-AUC: 0.5940987106035568

=== PROBABILIDADE POR CLASSE REAL ===
                 mean    median       min       max
classe_real                                        
0            0.227026  0.238572  0.134006  0.395089
1            0.255091  0.243895  0.134006  0.395089


### 2.2 Sensibilidade ao limiar de classificação (exploratória)

Os limiares abaixo ilustram o compromisso entre precisão e recall **no primeiro fold de validação**. **Não** são usados para selecionar o limiar da avaliação principal: isso exigiria validação interna nos dados de treino. Todas as comparações de EM abaixo utilizam o limiar pré-fixado de 0,5.

In [7]:
limiares_exploratorios = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
linhas_limiares = []
for limiar in limiares_exploratorios:
    previsto = (prob_proprio >= limiar).astype(int)
    linhas_limiares.append({
        "limiar": limiar,
        "acuracia": accuracy_score(yva, previsto),
        "precisao": precision_score(yva, previsto, zero_division=0),
        "recall": recall_score(yva, previsto, zero_division=0),
        "f1": f1_score(yva, previsto, zero_division=0),
        "positivos_previstos": int(previsto.sum()),
    })
tabela_limiares = pd.DataFrame(linhas_limiares)
display(tabela_limiares)
fig, ax = plt.subplots(figsize=(9, 5))
for metrica in ("acuracia", "precisao", "recall", "f1"):
    ax.plot(tabela_limiares["limiar"], tabela_limiares[metrica], marker="o", label=metrica)
ax.set(xlabel="Limiar de classificação", ylabel="Métrica",
       title="Bernoulli Naive Bayes — sensibilidade ao limiar (fold 0)")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(DIRETORIO_GRAFICOS / "sensibilidade_limiar_nb.png",
            dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("Gráfico salvo:", DIRETORIO_GRAFICOS / "sensibilidade_limiar_nb.png")

,limiar,acuracia,precisao,recall,f1,positivos_previstos
0,0.15,0.419716,0.249205,0.776554,0.377322,19803
1,0.20,0.504756,0.268329,0.687648,0.386025,16286
2,0.25,0.640814,0.314374,0.496617,0.385019,10039
3,0.30,0.689088,0.333427,0.373564,0.352356,7120
4,0.35,0.743596,0.338944,0.139418,0.197569,2614
5,0.40,0.773594,0.000000,0.000000,0.000000,0
6,0.50,0.773594,0.000000,0.000000,0.000000,0


Gráfico salvo: graficos_etapa4/sensibilidade_limiar_nb.png


## 3. Expectation-Maximization e perfis latentes

O EM é usado para ajustar misturas de gaussianas com **covariâncias diagonais**, separadamente para cada classe de FRP. A variável oculta \(Z\) indica o componente da mistura. Para manter a comparação controlada, o mesmo classificador em `em.py` é usado com:

- **K=1:** uma gaussiana diagonal por classe (Gaussian Naive Bayes);
- **K=2:** mistura com dois perfis latentes por classe;
- **K=3:** mistura com três perfis latentes por classe.

A transformação padronizada é ajustada apenas sobre o treino do fold. O uso de \(K>1\) flexibiliza \(P(X\mid Y)\), mas **não garante melhora** na classificação.

### 3.1 Experimento em um fold espacial

Primeiro avaliamos os modelos no fold 0, mantendo as mesmas classes, atributos e partições.

In [8]:
fold, treino, validacao = next(iterar_folds(dados, "espacial"))
p = preparar_particao(treino, validacao, "padronizada")
Xtr = p.X_treino.to_numpy(dtype=float)
Xva = p.X_avaliacao.to_numpy(dtype=float)
ytr = p.y_cls_treino.to_numpy()
yva = p.y_cls_avaliacao.to_numpy()

modelos = {}
linhas = []
for k in (1, 2, 3):
    m = NaiveBayesComEM(n_componentes=k, max_iter=100, tol=1e-6, reg_covar=1e-4, random_state=42)
    m.fit(Xtr, ytr)
    prob = m.predict_proba(Xva)[:, 1]
    pred = (prob >= 0.5).astype(int)
    modelos[k] = m
    linhas.append({
        "modelo": "Gaussian NB" if k == 1 else f"NB + EM ({k} perfis/classe)",
        "acuracia": accuracy_score(yva, pred),
        "precisao": precision_score(yva, pred, zero_division=0),
        "recall": recall_score(yva, pred, zero_division=0),
        "f1": f1_score(yva, pred, zero_division=0),
        "roc_auc": roc_auc_score(yva, prob),
        "positivos_previstos": int(pred.sum()),
    })
display(pd.DataFrame(linhas))

,modelo,acuracia,precisao,recall,f1,roc_auc,positivos_previstos
0,Gaussian NB,0.741922,0.389069,0.245319,0.300907,0.637251,4007
1,NB + EM (2 perfis/classe),0.751291,0.400824,0.199056,0.266008,0.635301,3156
2,NB + EM (3 perfis/classe),0.746232,0.395652,0.229111,0.290184,0.646786,3680


### 3.2 Curvas de convergência da log-verossimilhança

Cada curva registra a log-verossimilhança **de treino** de um modelo de mistura ajustado dentro de determinada classe. O índice 0 corresponde à inicialização. Curvas com números distintos de observações (classes 0 e 1) não devem ser comparadas apenas pela altura.

O gráfico é salvo **antes** de a figura ser fechada; não depende de `plt.show()` nem do renderizador Jupyter do VS Code.

In [9]:
fig, ax = plt.subplots(figsize=(10, 6))
resumo_convergencia = []
for k in (2, 3):
    for classe, mistura in zip(modelos[k].classes_, modelos[k].modelos_):
        historico = np.asarray(mistura.log_verossimilhanca_, dtype=float)
        diferencas = np.diff(historico)
        ax.plot(range(len(historico)), historico, marker=".", markersize=4,
                label=f"K={k}, classe={classe}")
        minimo = float(diferencas.min()) if len(diferencas) else np.nan
        resumo_convergencia.append({
            "K": k, "classe": int(classe), "iteracoes": mistura.n_iter_,
            "convergiu": mistura.convergiu_, "menor_variacao": minimo,
        })
        print(f"K={k}, classe={classe}: {mistura.n_iter_} iterações; "
              f"convergiu={mistura.convergiu_}; menor variação={minimo:.6f}")
        assert np.all(diferencas >= -1e-6 * (1 + np.abs(historico[:-1]))), (
            "Queda inesperada da log-verossimilhança"
        )

ax.set(xlabel="Iteração do EM (0 = inicialização)",
       ylabel="Log-verossimilhança de treino",
       title="Convergência do EM — primeiro fold espacial")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
caminho_convergencia = DIRETORIO_GRAFICOS / "log_verossimilhanca_em.png"
fig.savefig(caminho_convergencia, dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig)
display(pd.DataFrame(resumo_convergencia))
print("Gráfico salvo:", caminho_convergencia.resolve())

K=2, classe=0: 22 iterações; convergiu=True; menor variação=0.320359
K=2, classe=1: 9 iterações; convergiu=True; menor variação=0.035108
K=3, classe=0: 29 iterações; convergiu=True; menor variação=0.348859
K=3, classe=1: 32 iterações; convergiu=True; menor variação=0.069910


,K,classe,iteracoes,convergiu,menor_variacao
0,2,0,22,True,0.320359
1,2,1,9,True,0.035108
2,3,0,29,True,0.348859
3,3,1,32,True,0.069910


Gráfico salvo: /home/murillo-pc/Documentos/fia-queimadas-amazonia/graficos_etapa4/log_verossimilhanca_em.png


### 3.3 Interpretação descritiva dos perfis latentes

Os pesos estimam a frequência relativa dos componentes **dentro da classe**. As médias são apresentadas em **z-scores**: valores positivos estão acima da média de treino do atributo e negativos, abaixo. Os componentes representam padrões estatísticos; sua interpretação **não** estabelece causas ou categorias ambientais reais.

In [10]:
nomes_atributos = list(p.X_treino.columns)
for k in (2, 3):
    for classe, mistura in zip(modelos[k].classes_, modelos[k].modelos_):
        print(f"\nClasse {classe}, K={k} | Pesos dos perfis:", np.round(mistura.pesos_, 3))
        tabela = pd.DataFrame(mistura.medias_, columns=nomes_atributos)
        tabela.index = [f"perfil_{i+1}" for i in range(k)]
        display(tabela.round(3))


Classe 0, K=2 | Pesos dos perfis: [0.196 0.804]


,numero_dias_sem_chuva,precipitacao,lat,lon,mes,hora_utc
perfil_1,-0.565,0.937,0.199,0.026,0.021,-0.074
perfil_2,0.022,-0.191,0.026,0.001,0.004,0.040



Classe 1, K=2 | Pesos dos perfis: [0.114 0.886]


,numero_dias_sem_chuva,precipitacao,lat,lon,mes,hora_utc
perfil_1,-0.462,0.702,0.079,-0.197,0.406,-0.027
perfil_2,0.374,-0.191,-0.213,0.004,-0.079,-0.057



Classe 0, K=3 | Pesos dos perfis: [0.309 0.187 0.505]


,numero_dias_sem_chuva,precipitacao,lat,lon,mes,hora_utc
perfil_1,-0.513,-0.190,0.904,0.715,0.064,-0.680
perfil_2,-0.558,0.991,0.149,0.007,0.060,-0.049
perfil_3,0.336,-0.191,-0.490,-0.428,-0.046,0.470



Classe 1, K=3 | Pesos dos perfis: [0.11  0.204 0.686]


,numero_dias_sem_chuva,precipitacao,lat,lon,mes,hora_utc
perfil_1,-0.446,0.739,0.031,-0.223,0.424,0.001
perfil_2,-0.468,-0.190,0.849,0.638,-0.064,-0.859
perfil_3,0.616,-0.191,-0.519,-0.181,-0.083,0.177


## 4. Comparação final: Gaussian NB puro × NB com EM

Os três classificadores recebem os **mesmos atributos padronizados**, os **mesmos rótulos** e os **mesmos cinco folds espaciais**. A decisão usa limiar fixo de 0,5 em todos. O Bernoulli Naive Bayes da seção 2 é um experimento adicional, pois recebe atributos discretizados.

In [11]:
resultados = []
for fold, treino, validacao in iterar_folds(dados, "espacial"):
    p = preparar_particao(treino, validacao, "padronizada")
    Xtr = p.X_treino.to_numpy(dtype=float)
    Xva = p.X_avaliacao.to_numpy(dtype=float)
    ytr = p.y_cls_treino.to_numpy()
    yva = p.y_cls_avaliacao.to_numpy()
    for k in (1, 2, 3):
        m = NaiveBayesComEM(n_componentes=k, max_iter=100, tol=1e-6, reg_covar=1e-4, random_state=42).fit(Xtr, ytr)
        prob = m.predict_proba(Xva)[:, 1]
        pred = (prob >= 0.5).astype(int)
        resultados.append({"fold": fold, "K": k, "acuracia": accuracy_score(yva, pred),
                           "precisao": precision_score(yva, pred, zero_division=0),
                           "recall": recall_score(yva, pred, zero_division=0),
                           "f1": f1_score(yva, pred, zero_division=0),
                           "roc_auc": roc_auc_score(yva, prob)})
    print(f"Fold {fold} concluído")

resultados_em = pd.DataFrame(resultados)
display(resultados_em)
display(resultados_em.groupby("K")[["acuracia", "precisao", "recall", "f1", "roc_auc"]].agg(["mean", "std"]))

Fold 0 concluído
Fold 1 concluído
Fold 2 concluído
Fold 3 concluído
Fold 4 concluído


,fold,K,acuracia,precisao,recall,f1,roc_auc
0,0,1,0.741922,0.389069,0.245319,0.300907,0.637251
1,0,2,0.751291,0.400824,0.199056,0.266008,0.635301
2,0,3,0.746232,0.395652,0.229111,0.290184,0.646786
3,1,1,0.707827,0.412809,0.300220,0.347625,0.662143
4,1,2,0.724180,0.443165,0.248557,0.318486,0.659583
5,1,3,0.655884,0.356445,0.406156,0.379680,0.609622
6,2,1,0.693837,0.380138,0.381894,0.381014,0.656041
7,2,2,0.698824,0.376615,0.336702,0.355542,0.641068
8,2,3,0.643890,0.337738,0.461305,0.389967,0.609446
9,3,1,0.698315,0.373388,0.224520,0.280421,0.635743


acuracia            precisao              recall                  f1  \
       mean       std      mean       std      mean       std      mean   
K                                                                         
1  0.710658  0.018850  0.374871  0.034633  0.252354  0.100266  0.294669   
2  0.721453  0.020522  0.387985  0.037185  0.208410  0.097443  0.261278   
3  0.664771  0.046067  0.325122  0.056863  0.310238  0.114911  0.310327   

              roc_auc            
        std      mean       std  
K                                
1  0.083275  0.637096  0.026549  
2  0.091808  0.633058  0.020358  
3  0.071130  0.594808  0.040318

In [12]:
medias_em = resultados_em.groupby("K")[["acuracia", "precisao", "recall", "f1", "roc_auc"]].mean()
comparacao = medias_em.rename(index={1: "Gaussian NB (K=1)", 2: "NB + EM (K=2)", 3: "NB + EM (K=3)"})
display(comparacao.round(6))
diferencas_baseline = medias_em.loc[[2, 3]].subtract(medias_em.loc[1], axis="columns")
diferencas_baseline.index = ["K=2 menos K=1", "K=3 menos K=1"]
print("Diferenças absolutas frente ao Gaussian NB puro:")
display(diferencas_baseline.round(6))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(comparacao.index, comparacao["f1"])
ax.set(ylabel="F1-score médio", title="Classificação de FRP elevada — cinco folds espaciais")
ax.tick_params(axis="x", rotation=12)
fig.tight_layout()
caminho_f1 = DIRETORIO_GRAFICOS / "comparacao_f1_em.png"
fig.savefig(caminho_f1, dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("Gráfico salvo:", caminho_f1.resolve())

,acuracia,precisao,recall,f1,roc_auc
K,,,,,
Gaussian NB (K=1),0.710658,0.374871,0.252354,0.294669,0.637096
NB + EM (K=2),0.721453,0.387985,0.208410,0.261278,0.633058
NB + EM (K=3),0.664771,0.325122,0.310238,0.310327,0.594808


Diferenças absolutas frente ao Gaussian NB puro:


,acuracia,precisao,recall,f1,roc_auc
K=2 menos K=1,0.010795,0.013114,-0.043944,-0.033391,-0.004038
K=3 menos K=1,-0.045887,-0.049749,0.057884,0.015657,-0.042288


Gráfico salvo: /home/murillo-pc/Documentos/fia-queimadas-amazonia/graficos_etapa4/comparacao_f1_em.png


## 5. Discussão e conclusões

**MLE e MAP.** Como a classe `FRP elevada` é definida pelo percentil 75 da FRP no treino, sua frequência amostral é próxima de 25%. Nos folds analisados anteriormente, MLE e MAP Beta(2,2) diferiram muito pouco devido ao grande número de observações. O exemplo de amostra pequena ilustra como a priori Beta aproxima a estimativa de 0,5.

**Bernoulli Naive Bayes.** Nos experimentos anteriores, o limiar padrão 0,5 levou o classificador a prever somente a classe negativa, com recall e F1 iguais a zero. A checagem com a implementação do scikit-learn mostrou diferença numérica pequena, e a ROC-AUC do primeiro fold foi aproximadamente 0,594. Variar o limiar mostrou que há algum sinal discriminativo, mas os limiares testados na validação externa **não foram escolhidos como configuração final**.

**EM e perfis latentes.** O EM convergiu nos experimentos apresentados com dois e três componentes por classe, exibindo log-verossimilhança não decrescente e posterior estabilização. Os perfis diferem em suas médias padronizadas e pesos, mas a interpretação ambiental depende de análise adicional. O histórico completo é gerado na seção 3.2.

**Comparação preditiva.** Nos cinco folds espaciais anteriormente executados, o Gaussian NB puro (K=1) obteve F1 médio de **0,2947**, o modelo com K=2 obteve **0,2613**, e o modelo com K=3 obteve **0,3103**. Entretanto, a ROC-AUC média foi aproximadamente **0,6371**, **0,6331** e **0,5948**, respectivamente. Assim, K=3 apresentou pequeno ganho de F1 médio, mas pior discriminação segundo a ROC-AUC; **não houve melhora consistente em todas as métricas**. As tabelas produzidas na seção 4 são a fonte reproduzível desses resultados.

**Limitações.** O ajuste da verossimilhança não otimiza diretamente precisão, recall ou F1. Os modelos gaussianos usam uma hipótese de covariância diagonal, e os atributos disponíveis podem não capturar toda a variabilidade da FRP. Os resultados são válidos para o protocolo de 2024 apresentado; a avaliação independente em 2025 faz parte da integração final da equipe.

**Integração posterior.** As probabilidades de `predict_proba` podem ser consumidas pelas próximas etapas, desde que a equipe preserve a definição da classe, as transformações e as fronteiras treino/teste. Não há necessidade de retreinar com os dados reservados de 2025 nesta etapa.